In [8]:
import os
import torch
import numpy as np
import pyexr  # This will be our reliable loader
from torch.utils.data import Dataset, DataLoader

In [9]:
class KOREATECH_HolographyDataset(Dataset):
    def __init__(self, root_dir, encoder):
        self.encoder = encoder
        self.img_dir = os.path.join(root_dir, "img")
        self.depth_dir = os.path.join(root_dir, "depth")
        self.amp_dir = os.path.join(root_dir, "amp")
        self.phs_dir = os.path.join(root_dir, "phs")
        
        # Get all potential indices
        all_indices = sorted([f.replace(".exr", "") for f in os.listdir(self.img_dir) if f.endswith(".exr")])
        
        # Optional: You could pre-validate files here, but let's do it on-the-fly to save time
        self.indices = all_indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        
        # 1. Engineered Input (30 channels)
        x = self.encoder.encode(os.path.join(self.img_dir, f"{i}.exr"), 
                               os.path.join(self.depth_dir, f"{i}.exr"))
        
        # 2. Corrected Target Loading
        # Load Amplitude and Phase
        amp_raw = self.encoder.load_exr(os.path.join(self.amp_dir, f"{i}.exr"))
        phs_raw = self.encoder.load_exr(os.path.join(self.phs_dir, f"{i}.exr"))
        
        # Convert to Tensors and ensure [1, 512, 512]
        # pyexr/cv2 often returns [512, 512, 3] even for grayscale EXRs
        amp = torch.from_numpy(amp_raw)
        phs = torch.from_numpy(phs_raw)

        # Fix: Take only the first channel if it's 3-channel, and remove extra dims
        if amp.ndim == 3: amp = amp[:, :, 0] 
        if phs.ndim == 3: phs = phs[:, :, 0]
        
        # Final target stack [2, 512, 512]
        y = torch.cat([amp.unsqueeze(0), phs.unsqueeze(0)], dim=0)
            
        return x, y

In [10]:
# from huggingface_hub import snapshot_download

# # This will download ONLY the phs folder into your directory
# snapshot_download(
#     repo_id="SPIN-Lab/KOREATECH-CGH-512-3.6Mu",
#     repo_type="dataset",
#     local_dir=r"C:\Users\Kai Kumano\workspace\CGH-depth\dataset\KOREATECH-CGH-512-3.6Mu",
#     allow_patterns=["train/phs/*"]
# )
# print("Download complete!")

In [11]:
class KOREATECH_CGH_Encoder:
    def __init__(self, res=512, pitch=3.6e-6, wavelength=638e-9):
        self.res, self.pitch, self.wavelength = res, pitch, wavelength
        f = torch.fft.fftfreq(res, d=pitch)
        self.fy, self.fx = torch.meshgrid(f, f, indexing='ij')
        term = (1.0 / self.wavelength)**2 - self.fx**2 - self.fy**2
        self.phase_kernel_base = 2 * np.pi * torch.sqrt(torch.clamp(term, min=0))

    def load_exr(self, path):
        # pyexr loads directly into float32 numpy arrays 
        file = pyexr.open(path)
        return file.get().astype(np.float32)

    def encode(self, img_path, depth_path):
        rgb_raw = self.load_exr(img_path)
        depth_raw = self.load_exr(depth_path)

        # Correct shape for Torch [C, H, W]
        rgb = torch.from_numpy(rgb_raw).permute(2, 0, 1) if rgb_raw.ndim == 3 else torch.from_numpy(rgb_raw).unsqueeze(0)
        depth = torch.from_numpy(depth_raw).squeeze()
        
        # Physics: 20.33mm range
        z_map = depth * 20.3336e-3 
        encoding_arg = self.phase_kernel_base * z_map
        freq_cos, freq_sin = torch.cos(encoding_arg), torch.sin(encoding_arg)
        
        # 8 Depth Buckets (24 Channels)
        buckets = [rgb * ((depth >= i/8.0) & (depth < (i+1)/8.0)).float().unsqueeze(0) for i in range(8)]
            
        return torch.cat([rgb, depth.unsqueeze(0), freq_cos.unsqueeze(0), freq_sin.unsqueeze(0), *buckets], dim=0)

In [12]:
data_root = r"C:\Users\Kai Kumano\workspace\CGH-depth\dataset\KOREATECH-CGH-512-3.6Mu"
encoder = KOREATECH_CGH_Encoder()
train_dataset = KOREATECH_HolographyDataset(os.path.join(data_root, "train"), encoder)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

try:
    x, y = next(iter(train_loader))
    print(f"Success Input Shape: {x.shape}, Target Shape: {y.shape}")
except Exception as e:
    print(f"Final Attempt Failed: {e}")

Success Input Shape: torch.Size([1, 30, 512, 512]), Target Shape: torch.Size([1, 2, 512, 512])
